# sem01 с комментариями

Это копия семинара `regression/sem01.ipynb`, где к каждой строке кода добавлены комментарии,
а после каждой ячейки кода стоит блок **💬 Разбор**: что делает ячейка, что ты увидишь на выходе
и как это связано с лабой №1. Исходный текст семинара оставлен без изменений.

Места, которые в оригинале не работают, исправлены и помечены **🔧 ИСПРАВЛЕНО**:
загрузка данных через `!wget` (на Windows такой команды нет), устаревший `distplot`
и последние две ячейки про Lasso, где использовались неопределённые переменные.

## Линейная регрессия

На этом семинаре мы:
* Узнаем о том, что такое переобучение, что такое линейная регрессия и как не допустить переобучения в линейных моделях;
* Обучим линейную регрессию для предсказания цены дома;
* Научимся работать с разными типами признаков;
* Поймем, в чем отличие между разными регуляризаторами;
* Научимся пользоваться основными инструментами в `sklearn`: моделями, трансформерами и pipeline;
* Обсудим преобразования признаков и целевой переменной, которые могут помочь в обучении линейных моделей.

In [ ]:
import matplotlib.pyplot as plt   # базовая библиотека графиков
import seaborn as sns             # «надстройка» над matplotlib: красивее и короче
import pandas as pd               # таблицы (DataFrame)
import numpy as np                # массивы и математика
import warnings
warnings.simplefilter("ignore")   # прячем предупреждения, чтобы не засоряли вывод
%matplotlib inline

sns.set(style="darkgrid")         # тема оформления графиков: серый фон с сеткой

> 💬 **Разбор.** Импорты. Строка, начинающаяся с `%`, — это «магическая команда» Jupyter, а не Python:
> она просит рисовать графики прямо под ячейкой. В VS Code это работает и без неё.
> `warnings.simplefilter("ignore")` удобен на семинаре, но в своей лабе лучше его не ставить:
> предупреждения часто подсказывают, что что-то не так.

# Часть 0. Введение в линейные модели

Линейная регрессия &mdash; это модель следующего вида:
$$a(x) = \langle w, x \rangle + w_0$$
где $w \in \mathbb{R}^d$, $w_0 \in \mathbb{R}$. Обучить линейную регрессию &mdash; значит найти $w$ и $w_0$.

В машинном обучении часто говорят об *обобщающей способности модели*, то есть о способности модели работать на новых, тестовых данных хорошо. Если модель будет идеально предсказывать выборку, на которой она обучалась, но при этом просто ее запомнит, не "вытащив" из данных никакой закономерности, от нее будет мало толку. Такую модель называют *переобученной*: она слишком подстроилась под обучающие примеры, не выявив никакой полезной закономерности, которая позволила бы ей совершать хорошие предсказания на данных, которые она ранее не видела.

Рассмотрим следующий пример, на котором будет хорошо видно, что значит переобучение модели. Для этого нам понадобится сгенерировать синтетические данные. Рассмотрим зависимость $y(x) = \cos(1.5\pi x)$, $y$ &mdash; целевая переменная, а $x$ - объект (просто число от 0 до 1). В жизни мы наблюдаем какое-то конечное количество пар объект-таргет, поэтому смоделируем это, взяв 30 случайных точек $x_i$ в отрезке $[0;1]$. Более того, в реальной жизни целевая переменная может быть зашумленной (измерения в жизни не всегда точны), смоделируем это, зашумив значение функции нормальным шумом: $\tilde{y}_i = y(x_i) + \mathcal{N}(0, 0.01)$.

Попытаемся обучить три разных линейных модели: признаки для первой &mdash;  $\{x\}$, для второй &mdash; $\{x, x^2, x^3, x^4\}$, для третьей &mdash; $\{x, \dots, x^{20}\}$.

In [ ]:
from sklearn.linear_model import LinearRegression

np.random.seed(36)                         # фиксируем генератор случайных чисел -> результат повторяется
x = np.linspace(0, 1, 100)                 # 100 равномерных точек от 0 до 1 — для рисования «истинной» кривой
y = np.cos(1.5 * np.pi * x)                # истинная функция, которую модель НЕ знает

x_objects = np.random.uniform(0, 1, size=30)   # 30 случайных x — это наша «обучающая выборка»
y_objects = (np.cos(1.5 * np.pi * x_objects)   # их истинные значения
             + np.random.normal(scale=0.1, size=x_objects.shape))  # + шум, как в реальных измерениях

from sklearn.preprocessing import PolynomialFeatures
fig, axs = plt.subplots(figsize=(16, 4), ncols=3)   # три графика в ряд
for i, degree in enumerate([1, 4, 20]):             # три модели разной сложности
    # PolynomialFeatures превращает один признак x в набор x, x^2, ..., x^degree.
    # x_objects[:, None] делает из вектора формы (30,) таблицу формы (30, 1): sklearn ждёт 2D.
    X_objects = PolynomialFeatures(degree, include_bias=False).fit_transform(x_objects[:, None])
    X = PolynomialFeatures(degree, include_bias=False).fit_transform(x[:, None])
    regr = LinearRegression().fit(X_objects, y_objects)   # учим на 30 зашумлённых точках
    y_pred = regr.predict(X)                              # предсказываем на всей сетке, чтобы нарисовать линию
    axs[i].plot(x, y, label="Real function")              # синяя — правда
    axs[i].scatter(x_objects, y_objects, label="Data")    # точки — что видела модель
    axs[i].plot(x, y_pred, label="Prediction")            # оранжевая — что выучила модель
    if i == 0:
        axs[i].legend()
    axs[i].set_title("Degree = %d" % degree)
    axs[i].set_xlabel("$x$")
    axs[i].set_ylabel("$f(x)$")
    axs[i].set_ylim(-2, 2)                                # фиксируем ось, иначе степень 20 «улетит» за экран

> 💬 **Разбор.** Главная картинка про **недообучение и переобучение**.
> 
> - Степень 1 — прямая. Она не может повторить изгиб косинуса: модель слишком простая, это **недообучение**.
> - Степень 4 — почти совпадает с истинной функцией: сложность подходящая.
> - Степень 20 — проходит почти через каждую точку, но между точками и на краях её мотает:
>   она выучила **шум**, а не закономерность. Это **переобучение**: на train ошибка маленькая, на новых данных — большая.
> 
> Важный момент: все три модели — это `LinearRegression`. «Линейная» означает линейность по весам,
> а признаки могут быть любыми (x², x³…). Сложность модели здесь растёт за счёт числа признаков.
> 
> **Для лабы:** вопрос «что такое переобучение» есть в списке вопросов на защите. Эту картинку удобно пересказать.

# Часть 1. Загружаем данные

Мы будем работать с данными из соревнования [House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/overview), в котором требовалось предсказать стоимость жилья. Давайте сначала загрузим и немного изучим данные (можно взять `train.csv` со страницы соревнования и переименовать в `train_sem2.csv`).

🔧 **ИСПРАВЛЕНО.** В оригинале здесь `!wget ...`: это команда Linux, на Windows её нет.
Ниже то же самое на чистом Python. Если ссылка Dropbox не откроется, скачай `train.csv` со страницы
[соревнования Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)
(нужен вход в аккаунт), положи рядом с ноутбуком и переименуй в `train_sem2.csv`.

In [ ]:
import os, urllib.request

if not os.path.exists("train_sem2.csv"):          # качаем, только если файла ещё нет
    url = "https://www.dropbox.com/s/6dxq90t0prn2vaw/_train_sem2.csv?dl=1"   # dl=1 -> отдать сам файл, а не страницу
    try:
        urllib.request.urlretrieve(url, "train_sem2.csv")
        print("Скачано")
    except Exception as e:
        print("Не получилось скачать:", e, "\nСкачай train.csv с Kaggle вручную (см. текст выше).")
else:
    print("Файл уже есть")

In [ ]:
data = pd.read_csv("train_sem2.csv")   # читаем CSV в таблицу DataFrame
data.head()                            # первые 5 строк — быстро глянуть, что внутри

> 💬 **Разбор.** `head()` — первое, что делают с любыми данными. Строка таблицы — один дом, столбец — его характеристика,
> `SalePrice` — цена, её и предсказываем. Посреди таблицы стоят `...`: столбцов слишком много,
> pandas показывает не все (`pd.set_option("display.max_columns", 100)` покажет все).

In [ ]:
data.shape    # (число строк, число столбцов) -> (1460, 81)

In [ ]:
data.columns  # названия всех столбцов

> 💬 **Разбор.** 1460 домов и 81 столбец: `Id`, цена и 79 признаков. Описание каждого признака есть на странице соревнования на Kaggle.

In [ ]:
from sklearn.model_selection import train_test_split

data = data.drop(columns=["Id"])        # Id — просто номер строки, информации о доме в нём нет, выкидываем

y = data["SalePrice"]                   # целевая переменная (target) — то, что предсказываем
X = data.drop(columns=["SalePrice"])    # признаки — всё остальное. Target ОБЯЗАТЕЛЬНО убираем из X!

# 70% строк — на обучение, 30% — на проверку. random_state фиксирует, КАКИЕ строки куда попадут,
# чтобы при перезапуске получилось то же самое разбиение.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

> 💬 **Разбор.** **Разбиение на train и test.** На `X_train, y_train` модель учится. `X_test, y_test` мы прячем
> и используем только для проверки: так мы узнаём, как модель поведёт себя на домах, которых не видела.
> 
> Функция возвращает четыре объекта в строго таком порядке: `X_train, X_test, y_train, y_test`. Порядок легко перепутать.
> 
> **Для лабы:** у тебя то же самое, только `test_size=0.2` и `stratify=y`. В домах цена — непрерывное число,
> стратифицировать по нему нельзя. В вине `quality` — несколько целых значений, некоторые редкие,
> поэтому там `stratify` имеет смысл.

Посмотрим сначала на значения целевой переменной.

🔧 **ИСПРАВЛЕНО.** `sns.distplot` устарел и в новых версиях seaborn выдаёт ошибку или предупреждение. Замена — `sns.histplot(..., kde=True)`.

In [ ]:
sns.histplot(y_train, kde=True)   # гистограмма цен + сглаженная кривая плотности (kde)
plt.show()

> 💬 **Разбор.** Гистограмма: по оси X — цена, по оси Y — сколько домов в этом диапазоне. Распределение **скошено вправо**:
> большинство домов стоит 100–250 тыс., но есть хвост дорогих, до 755 тыс.
> 
> Почему это важно: модель учится минимизировать квадрат ошибки, а на дорогом доме ошибка в рублях
> (точнее, в долларах) огромная. Поэтому несколько дорогих домов могут «тянуть» модель на себя.
> Об этом следующая ячейка текста.
> 
> **Для лабы:** сделай такой же график для `quality` (у тебя он уже в шаблоне) и напиши, что видишь.

Судя по гистограмме, у нас есть примеры с нетипично большой стоимостью, что может помешать нам, если наша функция потерь слишком чувствительна к выбросам. В дальнейшем мы рассмотрим способы, как минимизировать ущерб от этого.

Так как для решения нашей задачи мы бы хотели обучить линейную регрессию, было бы хорошо найти признаки, "наиболее линейно" связанные с целевой переменной, иначе говоря, посмотреть на коэффициент корреляции Пирсона между признаками и целевой переменной. Заметим, что не все признаки являются числовыми, пока что мы не будем рассматривать такие признаки.

In [ ]:
numeric_data = X_train.select_dtypes([np.number])  # оставляем только числовые столбцы (текстовые типа "Pave" пока пропускаем)
numeric_data_mean = numeric_data.mean()            # среднее каждого числового столбца, ПОСЧИТАННОЕ ПО TRAIN
numeric_features = numeric_data.columns            # запоминаем список числовых признаков

# Заполняем пропуски (NaN) средними. Обрати внимание: для test тоже берём средние ПО TRAIN.
# Если бы считали по test или по всему датасету — это утечка информации из теста.
X_train = X_train.fillna(numeric_data_mean)
X_test = X_test.fillna(numeric_data_mean)

# Корреляция Пирсона каждого признака с ценой, от большей к меньшей
correlations = X_train[numeric_features].corrwith(y_train).sort_values(ascending=False)

plot = sns.barplot(y=correlations.index, x=correlations)   # горизонтальные столбики: признак -> корреляция
plot.figure.set_size_inches(15, 10)

> 💬 **Разбор.** Здесь три идеи.
> 
> 1. **Числовые и категориальные признаки.** Часть столбцов — текст (`Street`, `Neighborhood`).
>    Линейная модель умеет умножать только числа, поэтому пока работаем с числовыми.
>    Текстовые нужно кодировать, это делается в конце семинара (`OneHotEncoder`).
> 2. **Пропуски.** Модели sklearn не принимают `NaN`. Простейшее решение — заполнить средним.
>    Ключевой момент: среднее считается **только по train** и затем применяется к test. Это правило
>    «fit на train, применяем к test», и оно будет повторяться весь курс.
> 3. **Корреляция с target.** Наверху графика `OverallQual` (общее качество), `GrLivArea` (жилая площадь),
>    гаражи — разумно. Корреляция около 0 значит, что *линейной* связи нет, но нелинейная может быть.
> 
> **Для лабы:** в вине пропусков нет, но корреляции посчитать нужно. Важно написать не только цифры,
> но и что они означают.

Посмотрим на признаки из начала списка. Для этого нарисуем график зависимости целевой переменной от каждого из признаков. На этом графике каждая точка соответствует паре признак-таргет (такие графики называются scatter-plot).

In [ ]:
fig, axs = plt.subplots(figsize=(16, 5), ncols=3)
for i, feature in enumerate(["GrLivArea", "GarageArea", "TotalBsmtSF"]):   # три признака из верха списка корреляций
    axs[i].scatter(X_train[feature], y_train, alpha=0.2)   # каждая точка — один дом; alpha=0.2 — полупрозрачность, видно плотность
    axs[i].set_xlabel(feature)
    axs[i].set_ylabel("SalePrice")
plt.tight_layout()   # раздвигает графики, чтобы подписи не налезали

> 💬 **Разбор.** **Scatter plot** (диаграмма рассеяния) — признак по X, цена по Y. Видна «облачная» вытянутая вверх-вправо
> форма: больше площадь — выше цена. Также видны точки у нуля по X (нет гаража или подвала)
> и пара домов с огромной площадью, но низкой ценой — это **выбросы**, которые портят линейную модель.
> 
> Корреляция — одно число, scatter показывает форму связи целиком. Смотреть стоит на оба.

Видим, что между этими признаками и целевой переменной действительно наблюдается линейная зависимость.

# Часть 2. Первая модель

*Немного об обучении моделей*. В арсенале дата-саентиста кроме `pandas` и `matplotlib` должны быть библиотеки, позволяющие обучать модели. Для простых моделей (линейные модели, решающее дерево, ...) отлично подходит `sklearn`: в нем очень понятный и простой интерфейс. Несмотря на то, что в `sklearn` есть реализация бустинга и простых нейронных сетей, ими все же не пользуются и предпочитают специализированные библиотеки: `XGBoost`, `LightGBM` и пр. для градиентного бустинга над деревьями, `PyTorch`, `Tensorflow` и пр. для нейронных сетей. Так как мы будем обучать линейную регрессию, нам подойдет реализация из `sklearn`.

Попробуем обучить линейную регрессию на числовых признаках из нашего датасета. В `sklearn` есть несколько классов, реализующих линейную регрессию:
* [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) &mdash; "классическая" линейная регрессия с оптимизацией MSE. Веса находятся как точное решение: $w^* = (X^TX)^{-1}X^Ty$
* [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) &mdash; линейная регрессия с оптимизацией MSE и $\ell_2$-регуляризацией
* [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) &mdash; линейная регрессия с оптимизацией MSE и $\ell_1$-регуляризацией

**Почему Lasso зануляет часть коэффициентов, а Ridge обычно лишь сжимает их?** Разберём это на формулах и графиках в [лекционном блоке о L1 и L2](#l1-l2-lecture).

У моделей из `sklearn` есть методы `fit` и `predict`. Первый принимает на вход обучающую выборку и вектор целевых переменных и обучает модель, второй, будучи вызванным после обучения модели, возвращает предсказание на выборке. Попробуем обучить нашу первую модель на числовых признаках, которые у нас сейчас есть:

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error   # готовая функция RMSE (есть в sklearn >= 1.4)

model = Ridge()                                   # 1) создаём модель (alpha=1.0 по умолчанию)
model.fit(X_train[numeric_features], y_train)     # 2) обучаем: подбираются веса w и w0
y_pred = model.predict(X_test[numeric_features])          # 3) предсказываем на test
y_train_pred = model.predict(X_train[numeric_features])   #    и на train — для сравнения

print("Test RMSE = %.4f" % root_mean_squared_error(y_test, y_pred))
print("Train RMSE = %.4f" % root_mean_squared_error(y_train, y_train_pred))

> 💬 **Разбор.** Первая модель — цикл **создать → `fit` → `predict` → оценить**. Он одинаков для всех моделей sklearn,
> в лабе ты повторишь его для LinearRegression, RandomForest и GradientBoosting.
> 
> `Ridge` — линейная регрессия с регуляризацией (штрафом за большие веса, см. конец семинара).
> 
> RMSE здесь в долларах: около 32 тыс. на test при средней цене 180 тыс.
> 
> Обычно ошибка на train меньше, чем на test, потому что train модель видела. Здесь вышло наоборот
> (train ≈ 35.6 тыс., test ≈ 32.1 тыс.). Это не ошибка: в train случайно попали самые дорогие дома-выбросы,
> на которых модель сильно промахивается, а квадрат ошибки их раздувает. Такое бывает при небольших данных,
> и именно поэтому одной оценки мало — дальше идёт кросс-валидация.
> Признак переобучения — наоборот: train сильно лучше test.

Мы обучили первую модель и даже посчитали ее качество на отложенной выборке! Давайте теперь посмотрим на то, как можно оценить качество модели с помощью кросс-валидации. 

При кросс-валидации мы делим обучающую выборку на $n$ частей (fold). Затем мы обучаем $n$ моделей: каждая модель обучается при отсутствии соответствующего фолда, то есть $i$-ая модель обучается на всей обучающей выборке, кроме объектов, которые попали в $i$-ый фолд (out-of-fold). Затем мы измеряем качество $i$-ой модели на $i$-ом фолде. Так как он не участвовал в обучении этой модели, мы получим "честный результат". После этого, для получения финального значения метрики качества, мы можем усреднить полученные нами $n$ значений.

In [ ]:
from sklearn.model_selection import cross_val_score

# 10-фолдовая кросс-валидация ТОЛЬКО на train: 10 раз учим на 9/10 данных и проверяем на оставшейся 1/10
cv_scores = cross_val_score(model, X_train[numeric_features], y_train, cv=10, scoring="neg_root_mean_squared_error")
print("Cross validation scores:\n\t", "\n\t".join("%.4f" % x for x in cv_scores))   # 10 значений, по одному на фолд
print("Mean CV RMSE = %.4f" % np.mean(-cv_scores))   # минус, т.к. sklearn вернул отрицательные числа (см. ниже)

> 💬 **Разбор.** **Кросс-валидация** даёт оценку качества, не трогая test. Она нужна для **выбора**: какую модель взять,
> какой `alpha` поставить. Test остаётся для одной финальной проверки.
> 
> Посмотри на 10 значений: они заметно различаются от фолда к фолду. Значит, оценка по одному разбиению
> шумная, и усреднение по фолдам надёжнее.
> 
> (В оригинале строка подписана «Mean CV MSE», но считается RMSE — я исправил подпись.)
> 
> **Для лабы:** в шаблоне так подбираются параметры бустинга (`GridSearchCV(..., cv=5)` внутри делает именно это).

Обратите внимание на то, что результаты `cv_scores` получились отрицательными. Это соглашение в `sklearn` (скоринговую функцию нужно максимизировать). Поэтому все стандартные скореры называются `neg_*`, например, `neg_root_mean_squared_error`.

Обратите внимание, что по отложенной выборке и при кросс-валидации мы считаем RMSE (Root Mean Squared Error), хотя в функционале ошибки при обучении модели используется MSE.

$$\text{RMSE}(X, y, a) = \sqrt{\frac{1}{\ell}\sum_{i=1}^{\ell} (y_i - a(x_i))^2}$$

Если бы RMSE не входил в стандартные метрики `sklearn`, можно было бы его определить через функцию `sklearn.metrics.make_scorer`.

In [ ]:
from sklearn.metrics import make_scorer

def rmse(y_true, y_pred):
    error = (y_true - y_pred) ** 2      # квадрат ошибки на каждом объекте
    return np.sqrt(np.mean(error))      # среднее -> корень

rmse_scorer = make_scorer(
    rmse,
    greater_is_better=False             # говорим sklearn: меньше — лучше, поэтому он будет хранить значение со знаком минус
)

> 💬 **Разбор.** Та же RMSE, написанная вручную по формуле выше. Полезно, чтобы убедиться, что формулу ты понимаешь,
> и чтобы знать: любую свою метрику можно подключить к sklearn через `make_scorer`.

In [ ]:
cv_scores = cross_val_score(model, X_train[numeric_features], y_train, cv=10, scoring=rmse_scorer)
print("Cross validation scores:\n\t", "\n\t".join("%.4f" % x for x in cv_scores))
print("Mean CV RMSE = %.4f" % np.mean(-cv_scores))

> 💬 **Разбор.** Числа должны совпасть с предыдущей ячейкой — значит, наша функция `rmse` считает то же самое, что встроенная.

Для того, чтобы иметь некоторую точку отсчета, удобно посчитать оптимальное значение функции потерь при константном предсказании.

In [ ]:
best_constant = y_train.mean()   # лучшая константа для RMSE — среднее target по TRAIN
# best_constant * np.ones(...) — массив, где для каждого дома предсказана одна и та же цена
print("Test RMSE with best constant = %.4f" % root_mean_squared_error(y_test, best_constant * np.ones(y_test.shape)))
print("Train RMSE with best constant = %.4f" % root_mean_squared_error(y_train, best_constant * np.ones(y_train.shape)))

> 💬 **Разбор.** Это **baseline** — тот самый, который обязателен в лабе. Предсказываем всем домам одну и ту же цену.
> Почему среднее: именно среднее минимизирует сумму квадратов отклонений, а значит и RMSE.
> 
> Сравни с Ridge выше: RMSE константы порядка 75–80 тыс., у Ridge примерно вдвое меньше. Модель действительно
> извлекает информацию из признаков. Если бы модель была не лучше константы — она бесполезна.
> 
> **Для лабы:** у тебя ровно эта строка, только для `quality`.

Давайте посмотрим на то, какие же признаки оказались самыми "сильными". Для этого визуализируем веса, соответствующие признакам. Чем больше вес &mdash; тем более сильным является признак.

In [ ]:
def show_weights(features, weights, scales):
    fig, axs = plt.subplots(figsize=(14, 10), ncols=2)           # два графика рядом
    sorted_weights = sorted(zip(weights, features, scales), reverse=True)   # склеиваем тройки (вес, имя, масштаб) и сортируем по весу
    weights = [x[0] for x in sorted_weights]                      # и распаковываем обратно в три списка
    features = [x[1] for x in sorted_weights]
    scales = [x[2] for x in sorted_weights]
    sns.barplot(y=features, x=weights, ax=axs[0])                 # слева — веса модели
    axs[0].set_xlabel("Weight")
    sns.barplot(y=features, x=scales, ax=axs[1])                  # справа — стандартное отклонение признака
    axs[1].set_xlabel("Scale")
    plt.tight_layout()

In [ ]:
show_weights(numeric_features, model.coef_, X_train[numeric_features].std())   # coef_ — выученные веса w

> 💬 **Разбор.** **Главная ловушка с коэффициентами.** Посмотри на два графика вместе: у признаков с маленьким разбросом
> (например, `GarageCars` — 0..4 машины) веса огромные, а у признаков с большим разбросом (`LotArea` —
> площадь участка в квадратных футах, десятки тысяч) веса крошечные.
> 
> Причина: вес — это «сколько долларов за единицу признака». Одна машина в гараже и один квадратный фут —
> несопоставимые единицы. Поэтому **по сырым весам нельзя судить, какой признак важнее**.
> 
> Это ровно тот вопрос, который задан в твоей лабе: «сравнивать абсолютные значения коэффициентов признаков
> в разных масштабах некорректно». Решение — в следующей ячейке.

Будем масштабировать наши признаки перед обучением модели. Это, среди, прочего, сделает нашу регуляризацию более честной: теперь все признаки будут регуляризоваться в равной степени.

Для этого воспользуемся трансформером [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html). Трансформеры в `sklearn` имеют методы `fit` и `transform` (а еще `fit_transform`). Метод `fit` принимает на вход обучающую выборку и считает по ней необходимые значения (например статистики, как `StandardScaler`: среднее и стандартное отклонение каждого из признаков). `transform` применяет преобразование к переданной выборке.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numeric_features])  # fit: запоминает среднее и std каждого признака ПО TRAIN; transform: (x - mean) / std
X_test_scaled = scaler.transform(X_test[numeric_features])        # для test ТОЛЬКО transform, с теми же mean/std от train

model = Ridge()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_train_pred = model.predict(X_train_scaled)

print("Test RMSE = %.4f" % root_mean_squared_error(y_test, y_pred))
print("Train RMSE = %.4f" % root_mean_squared_error(y_train, y_train_pred))

> 💬 **Разбор.** **Стандартизация**: из каждого признака вычитаем среднее и делим на стандартное отклонение.
> После этого у всех признаков среднее 0 и разброс 1 — масштабы сравнимы.
> 
> Самое важное правило ноутбука: `fit_transform` — на train, `transform` — на test.
> Если сделать `fit` на test или на всём датасете, в обучение попадёт информация о тесте — **утечка**,
> в README курса это прямо названо критической ошибкой.
> 
> RMSE изменился совсем чуть-чуть (здесь alpha=1 — слабый штраф). У обычной `LinearRegression` масштабирование качество не меняет вообще,
> а у `Ridge` меняет: штраф `alpha * сумма w²` зависит от величины весов, а она зависит от масштаба.

In [ ]:
scales = pd.Series(data=X_train_scaled.std(axis=0), index=numeric_features)   # после scaler все std = 1
show_weights(numeric_features, model.coef_, scales)

> 💬 **Разбор.** Теперь правый график — сплошные единицы, и веса слева можно сравнивать: вес = на сколько долларов меняется
> цена при росте признака на одно стандартное отклонение. Наверху теперь `OverallQual` и `GrLivArea` —
> совпадает со здравым смыслом и с корреляциями выше.
> 
> **Для лабы:** у тебя в шаблоне `Pipeline(StandardScaler, LinearRegression)` и график `coef_` — это то же самое.

Наряду с параметрами (веса $w$, $w_0$), которые модель оптимизирует на этапе обучения, у модели есть и гиперпараметры. У нашей модели это `alpha` &mdash; коэффициент регуляризации. Подбирают его обычно по сетке, измеряя качество на валидационной (не тестовой) выборке или с помощью кросс-валидации. Посмотрим, как это можно сделать (заметьте, что мы перебираем `alpha` по логарифмической сетке, чтобы узнать оптимальный порядок величины).

In [ ]:
from sklearn.model_selection import GridSearchCV

alphas = np.logspace(-2, 3, 20)    # 20 значений от 10^-2 до 10^3, равномерно по логарифмической шкале
searcher = GridSearchCV(Ridge(), [{"alpha": alphas}], scoring="neg_root_mean_squared_error", cv=10)
searcher.fit(X_train_scaled, y_train)   # для КАЖДОГО alpha — 10-фолдовая кросс-валидация на train

best_alpha = searcher.best_params_["alpha"]   # alpha с лучшей средней CV-ошибкой
print("Best alpha = %.4f" % best_alpha)

plt.plot(alphas, -searcher.cv_results_["mean_test_score"])   # средняя CV-RMSE для каждого alpha
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("CV score")

> 💬 **Разбор.** **Параметры и гиперпараметры.** Веса `w` модель находит сама при `fit`. А `alpha` (сила штрафа) мы задаём
> до обучения — это гиперпараметр. Подбираем его перебором: `GridSearchCV` для каждого значения из сетки
> делает кросс-валидацию и выбирает лучшее. Test при этом не используется.
> 
> Сетка логарифмическая (0.01, 0.02, … 1000), потому что сначала важно угадать порядок величины.
> 
> На графике кривая сначала идёт вниз (регуляризация помогает), потом вверх (штраф слишком сильный,
> веса сжаты к нулю — недообучение). Минимум и есть `best_alpha`.
> 
> **Для лабы:** в шаблоне так же подбираются `n_estimators`, `learning_rate`, `max_depth` бустинга.

Попробуем обучить модель с подобранным коэффициентом регуляризации. Заодно воспользуемся очень удобным классом [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): обучение модели часто представляется как последовательность некоторых действий с обучающей и тестовой выборками (например, сначала нужно отмасштабировать выборку (причем для обучающей выборки нужно применить метод `fit`, а для тестовой &mdash; `transform`), а затем обучить/применить модель (для обучающей `fit`, а для тестовой &mdash; `predict`). `Pipeline` позволяет хранить эту последовательность шагов и корректно обрабатывает разные типы выборок: и обучающую, и тестовую.

In [ ]:
from sklearn.pipeline import Pipeline

simple_pipeline = Pipeline([
    ('scaling', StandardScaler()),       # шаг 1: масштабирование (имя шага — любое)
    ('regression', Ridge(best_alpha))    # шаг 2: модель с подобранным alpha
])

model = simple_pipeline.fit(X_train[numeric_features], y_train)   # на train: scaler.fit_transform -> ridge.fit
y_pred = model.predict(X_test[numeric_features])                   # на test: scaler.transform -> ridge.predict
print("Test RMSE = %.4f" % root_mean_squared_error(y_test, y_pred))

> 💬 **Разбор.** **Pipeline** склеивает шаги в один объект с обычными `fit` и `predict`. Внутри он сам вызывает
> `fit_transform` на train и только `transform` на test, поэтому ошибиться и устроить утечку становится сложно.
> Ещё он правильно работает внутри кросс-валидации: scaler переобучается на каждом фолде.
> 
> Это итоговая модель первой части семинара. В твоём шаблоне ровно такая конструкция, только с `LinearRegression`.

## Почему L1 зануляет веса, а L2 обычно только уменьшает их

После Ridge сравним две регуляризации:

$$L_2 = \alpha \sum_j w_j^2, \qquad L_1 = \alpha \sum_j |w_j|.$$

**Ridge (L2)** штрафует большие коэффициенты и обычно плавно уменьшает их. **Lasso (L1)** может доводить часть коэффициентов точно до нуля, поэтому его можно рассматривать в том числе как способ отбора признаков.

Практически важно помнить две вещи: сила регуляризации задаётся `alpha`, а перед Ridge/Lasso числовые признаки нужно масштабировать, иначе признаки разных масштабов штрафуются несопоставимо.

### L1 и L2: что запомнить

- `Ridge` = L2: коэффициенты сжимаются, но обычно остаются ненулевыми.
- `Lasso` = L1: часть коэффициентов может стать нулевой.
- Большой `alpha` усиливает регуляризацию; слишком большой может привести к недообучению.
- Сравнивать модели нужно по качеству на новых данных, а не по ошибке на train.

🔧 **ИСПРАВЛЕНО.** В оригинале следующие две ячейки не запускались: не были импортированы `ColumnTransformer`
и `OneHotEncoder`, не определён список `categorical`, а во второй ячейке используется несуществующая
переменная `pipeline`. Кроме того, Ridge (только числовые признаки) сравнивался с Lasso (все признаки) —
нечестное сравнение. Ниже исправленная версия: обе модели на одинаковых признаках.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical = X_train.select_dtypes(exclude=[np.number]).columns   # текстовые (категориальные) столбцы
X_train[categorical] = X_train[categorical].fillna("missing")      # пропуск в категории — тоже категория
X_test[categorical] = X_test[categorical].fillna("missing")

# ColumnTransformer применяет разные преобразования к разным столбцам и склеивает результат
column_transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown="ignore"), categorical),   # текст -> набор столбцов 0/1
    ('scaling', StandardScaler(), numeric_features)                 # числа -> стандартизация
])

ridge_pipeline = Pipeline(steps=[
    ('ohe_and_scaling', column_transformer),
    ('regression', Ridge(best_alpha))
])
lasso_pipeline = Pipeline(steps=[
    ('ohe_and_scaling', column_transformer),
    ('regression', Lasso(alpha=100, max_iter=10000))   # alpha подобран под масштаб цен в долларах
])

for name, pipe in [("Ridge", ridge_pipeline), ("Lasso", lasso_pipeline)]:
    pipe.fit(X_train, y_train)
    print("%s test RMSE = %.4f" % (name, root_mean_squared_error(y_test, pipe.predict(X_test))))

> 💬 **Разбор.** **One-hot encoding.** Признак `Street` со значениями `Pave`/`Grvl` превращается в два столбца
> `Street_Pave` и `Street_Grvl` из нулей и единиц. Так текст становится числами, с которыми умеет работать
> линейная модель. `handle_unknown="ignore"` — если в test встретится значение, которого не было в train,
> не падать, а поставить нули.
> 
> `ColumnTransformer` позволяет в одном объекте сказать: эти столбцы кодируем, эти масштабируем.
> Вместе с Pipeline получается полная обработка «сырые данные → предсказание».
> 
> После добавления категориальных признаков ошибка обычно падает: в них есть полезная информация
> (район, тип отделки и т.п.).
> 
> **Для лабы:** в вине категориальных признаков нет (`is_red` уже число 0/1), так что эта часть тебе не нужна.

In [ ]:
ridge_coef = ridge_pipeline.named_steps["regression"].coef_   # веса Ridge
lasso_coef = lasso_pipeline.named_steps["regression"].coef_   # веса Lasso
print("Всего признаков после one-hot:", len(ridge_coef))
print("Zero weights in Ridge:", np.sum(ridge_coef == 0))
print("Zero weights in Lasso:", np.sum(lasso_coef == 0))

> 💬 **Разбор.** Иллюстрация разницы L1 и L2: у Ridge нулевых весов нет (веса только сжаты), а у Lasso заметная часть
> весов ровно 0 — модель сама «выкинула» эти признаки. Поэтому Lasso используют и как способ отбора признаков.
> 
> `named_steps["regression"]` — так достают конкретный шаг из Pipeline по имени, которое мы дали при создании.

# Итог семинара

Сегодняшняя практическая линия: **Linear Regression → масштабирование → Ridge/L2 → Lasso/L1**.

При работе с линейной регрессией смотрим не только на `fit/predict`, но и на RMSE, разницу между train и test, масштаб признаков и коэффициенты модели. Категориальные признаки, сложный feature engineering, биннинг и преобразование target сознательно оставлены за пределами этого короткого семинара.

## Что из семинара нужно для лабы №1

| Из семинара | В лабе |
| --- | --- |
| `train_test_split(..., random_state=...)` | разбиение, плюс `stratify=y` |
| `best_constant = y_train.mean()` | baseline |
| `root_mean_squared_error` | RMSE (плюс `mean_absolute_error` для MAE) |
| `StandardScaler` + `Pipeline` | линейная регрессия |
| `show_weights` / `coef_` после масштабирования | анализ коэффициентов |
| `cross_val_score`, `GridSearchCV` | подбор параметров бустинга без теста |
| картинка с полиномами | ответ на вопрос «что такое переобучение» |

Не нужно: one-hot (в вине нет текстовых признаков), Lasso/Ridge (не обязательны, но можно попробовать
Ridge вместо LinearRegression и сравнить).